In [94]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from elasticsearch import Elasticsearch
import pandas as pd
import json
import sqlite3
import pyodbc
from GetWorkFlow.db import DB
from elasticsearch import helpers
import requests

from elastic_settings import (
    all_cols,
    nested_cols,
    query_fields,
    INDEX_NAME,
    mappings,
    make_request_body,
    get_auth_header,
    make_elastic_query
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [95]:
from datetime import datetime, timedelta, timezone
(datetime.now(timezone.utc) - timedelta(minutes=1)).strftime('%Y-%m-%dT%H:%M:%S')

'2022-05-11T18:43:18'

### Credentials

In [96]:
from all_envs import *

In [97]:
spvars = test_SPVars()

SPVars(base_url='https://operationcentre.ms.bell.ca', env='PROD', client_secret='845e3004-c0f2-41c7-845f-2dd89481c842', user='svc-oc-smarttask-a', pw='bwx6BTQ!uqe-dxn2vtk')


In [124]:
def get_service_user_OC_token(sp_vars):
    """Post request to generate authentication token"""

    url = f"{sp_vars.base_url}/auth/realms/oc/protocol/openid-connect/token"

    payload=f"grant_type=password&client_id=oc-backend&"+\
            f"client_secret={sp_vars.client_secret}&"+\
            f"username={sp_vars.user}&password={sp_vars.pw}"
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Cookie': 'db4fdfa0d404bc10acda4670ccb15825=2a50a3c37d5e6020cba6f3bc9a4833a3; e69f1ea65bfd1ce3ad5996309f24ac3e=263f5d6acf2eb274a1d87c25ac4c33ad'
    }

    return requests.request("POST", url, headers=headers, data=payload, verify=False)


def smarttask_api_search_request(sp_vars, auth_header, date, limit=500):
    """Search the API based on payload found in elastic_settings.py
    The date in this payload is dynamic"""

    url = f"{sp_vars.base_url}/api/service-order/smarttask/order/_search?offset=0&limit={limit}&fields="

    payload = json.dumps(make_request_body(date=date))
    headers = get_auth_header(auth_header)

    return requests.request("POST", url, headers=headers, data=payload, verify=False)


In [125]:
def start_scroll_elastic_requests(sp_vars, auth_header, date, limit):

  url = f"{sp_vars.base_url}/api/service-order/smarttask/order/_startScroll?offset=0&limit={limit}&fields="

  payload = json.dumps(make_request_body(date=date))
  headers = get_auth_header(auth_header)

  return requests.request("POST", url, headers=headers, data=payload, verify=False)

In [126]:
def continue_scroll_elastic_requests(sp_vars, auth_header, date, scroll_id):

  url = f"{sp_vars.base_url}/api/service-order/smarttask/order/_continueScroll?scrollId={scroll_id}"

  payload = json.dumps(make_request_body(date=date))
  headers = get_auth_header(auth_header)

  return requests.request("POST", url, headers=headers, data=payload, verify=False)

In [127]:

service_token = get_service_user_OC_token(sp_vars)
access_token = json.loads(service_token.text)['access_token']


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


### Scroll Tests

In [10]:
index_name = 'bbm_aiml_stm_prod_test_2'

from initialize_elastic_index import prepare_data_and_load_to_index
from elastic_api_index import initialize_elastic_connection

es_connection = initialize_elastic_connection()

service_token = get_service_user_OC_token(sp_vars)
access_token = json.loads(service_token.text)['access_token']
start_response = start_scroll_elastic_requests(sp_vars, access_token, "2022-05-08T12:03:41", 500)
scroll_id = start_response.headers['x-scroll-id']

prepare_data_and_load_to_index(es_connection, 
                                json.loads(start_response.text)['results'],
                                index_name)

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\elasticsearch\connection\http_urllib3.py:209: UserWarning: Connecting to https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-package

Finished indexing data to Elastic


In [11]:
import time
while True:
    start_time = time.time()

    next_scroll_response_raw = continue_scroll_elastic_requests(
            sp_vars, access_token, "2022-05-08T12:03:41", scroll_id
        )
    next_scroll_response_text = json.loads(next_scroll_response_raw.text)

    try:
        next_scroll_results = next_scroll_response_text['results']
    except:
        print(next_scroll_response_text)
        break
    prepare_data_and_load_to_index(es_connection, next_scroll_results, index_name)
    
    print(f"---{time.time() - start_time} seconds ---")

    if not next_scroll_results:
        break

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Finished indexing data to Elastic
---12.268925189971924 seconds ---


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Finished indexing data to Elastic
---12.434939861297607 seconds ---


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


...

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Finished indexing data to Elastic
---15.34616208076477 seconds ---


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Finished indexing data to Elastic
---3.2192413806915283 seconds ---
Empty dict - Nothing was indexed
---0.21305012702941895 seconds ---


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [35]:
l_start = json.loads(r.text)['results']
l_cont = json.loads(c.text)['results']

In [66]:
# Test for duplicate request_ids
from collections import Counter
max(Counter([d['id'] for d in l_start]).values())

2

### Mapping & indexing tests

What we are testing:

1. For each flattened field, we cannot use 'nan' values when there is no data. If there is no data for a flattened field, we must represnt the fact that there is no data by a dictionary. 
2. Some requests have nested data that other requests don't have. Therefore, subfields names can be different from one request to the next. Does this difference get appropriatly indexed in Elastic?
3. When the same flattend field has different nested levels (because of 1 and 2), does this difference get appropriatly indexed in Elastic?  


In [128]:
r = smarttask_api_search_request(sp_vars, access_token, "2020-04-27T10:20:55")

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'operationcentre.ms.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [108]:
r.text

'{"metadata":{"offset":0,"limit":500,"totalRecordCount":18302},"results":[{"id":"bsd_REQ-545588","lastUpdated":"2022-05-11T18:45:47.912101108Z","accessPolicyTag":["DIVISION_RETAIL"],"workOrder":{"id":"bsd_REQ-545588","status":"new","controlDesk":"managedServicesSPOA","group":"Queue not found","tags":["CREATED_FROM_EMAIL_REQUEST","AIML_PROCESSED","SMARTPATH_DISTRIBUTION_MODE_QUEUE"]},"source":{"id":"bsd_REQ-545588","externalId":"REQ-545588","emailSubject":"SR886118 - Please be advised that a new worklog has been created./ Veuillez remarquer qu\'une entrée du journal des travaux a été créée.","orderDate":"2022-05-11T18:45:43Z","requestedStartDate":"2022-05-11T18:45:43Z","lastUpdated":"2022-05-11T18:45:47.910303932Z","referredType":"ServiceOrder","sourceSystemId":"BSD","requestType":"generalInquiry-othergeneral","requestSource":"email","customerSupportModel":"Standard","businessUnit":"Retail","incomingMailbox":"cibcmac@bell.ca","status":"acknowledged","product":"managedServices","serviceR

In [129]:
r_dict = json.loads(r.text)

In [130]:
meta_data = r_dict['metadata']
results = r_dict['results']

In [131]:
len(results)

500

In [132]:
meta_data

{'offset': 0, 'limit': 500, 'totalRecordCount': 18256}

In [136]:

results[5]['workOrder']['status']

'new'

In [6]:
f = open('../tests/dummy_smartpath_response.json')
r_dict = json.load(f)
results = r_dict['results']

In [68]:
def nested_col_nans_to_dict(results, nested_cols):
    """For each flattened field, we cannot use 'nan' values when there is no data. If there is no data
    for a flattened field, we must represent the fact that there is no data by a dictionary"""

    for resp in results:
        for nested_col in nested_cols:
            if nested_col not in resp.keys():
                resp.update({nested_col: {"value": "no-data"}})
    return results

Current project environment: PROD
Current SP environment: PROD
https://operationcentre.ms.bell.ca
845e3004-c0f2-41c7-845f-2dd89481c842


In [69]:
def prepare_data_for_elastic(results, all_cols):
    """
    1. Make custom nan values for flattened fields for indexation puposes
    2. Append foc targets to API results
    3. Then create a dataframe that is converted to a list of dicts. This format is
    requred for Elasticsearch indexation. Each dictionnary will become an Elastci doc"""

    # 1
    results = nested_col_nans_to_dict(results, nested_cols)

    # 2
    # con = sqlite3.connect(database)
    # cur = con.cursor()
    
    con = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+SQL_SERVER+';DATABASE='+DATABASE+';UID='+SQL_USER+';PWD='+SQL_PASS)
    cur = con.cursor()

    for res in results:
        foc_target = get_foc_target(cur, res)
        res['source']['focTarget'] = foc_target
        res['request_id'] = res['id']
        # res['lastUpdated'] = res['lastUpdated'].replace('Z', ' ')
        # res['lastUpdated'] = res['lastUpdated'][:-1] + 'Z'
        del res['id']

    # 3
    df = pd.DataFrame(data=results, columns=all_cols)
    df = df.fillna('nan')

    return df.to_dict('records')

In [70]:
def get_foc_target(cur, res):
    """Assumption: if either of the columns to retrieve an foc target from the db is None,
    we assume the foctarget is 0"""

    sql_query_values = {}
    for col in ['requestSource', 'product', 'serviceRegion', 'requestType']:
        if col in res['source'].keys():
            sql_query_values[col] = res['source'][col].lower()
        else:
            sql_query_values[col] = None

    if not all(
        (isinstance(sql_query_values['requestSource'], str),
         isinstance(sql_query_values['product'], str),
         isinstance(sql_query_values['serviceRegion'], str),
         isinstance(sql_query_values['requestType'], str))
         ):
        return 0

    sql = f"""SELECT CASE WHEN foc_target IS NULL THEN 0 ELSE foc_target 
             END as focTarget FROM {DB.foc_targets()}
    WHERE request_source='""" + sql_query_values['requestSource'] + """'
    AND product='""" + sql_query_values['product'] + """'
    AND service_region='""" + sql_query_values['serviceRegion'] + """'
    AND request_type='""" + sql_query_values['requestType'] + "'"

    foc_target = cur.execute(sql).fetchall()
    
    if not foc_target:
        return 0
    return foc_target[0][0]

In [71]:
df_dict = prepare_data_for_elastic(results, all_cols)

In [72]:
df_dict

[{'request_id': 'bsd_REQ-544756',
  'lastUpdated': '2022-05-11T15:26:36.538190233Z',
  'accessPolicyTag': ['DIVISION_RETAIL'],
  'OrderResponse': {'value': 'no-data'},
  'workOrder': {'id': 'bsd_REQ-544756',
   'status': 'inProgress',
   'controlDesk': 'bbscL1',
   'assignee': {'id': 'ba0cdrq', 'name': 'Marc Larivee'},
   'tags': ['CREATED_FROM_SMARTPATH', 'SMARTPATH_DISTRIBUTION_MODE_FILTER']},
  'source': {'id': 'bsd_REQ-544756',
   'externalId': 'REQ-544756',
   'orderDate': '2022-05-11T15:26:28Z',
   'requestedStartDate': '2022-05-11T15:26:28Z',
   'startDate': '2022-05-11T15:26:33.226889392Z',
   'lastUpdated': '2022-05-11T15:26:36.536276519Z',
   'referredType': 'ServiceOrder',
   'sourceSystemId': 'BSD',
   'requestType': 'change',
   'requestSource': 'smartpath',
   'billingAccount': '506155943',
   'customerSupportModel': 'Standard',
   'businessUnit': 'Retail',
   'crdDdst': 'AAO3377',
   'status': 'inProgress',
   'product': 'businessInternet',
   'serviceRegion': 'ON',
   '

### Flattened

In [73]:

def initialize_elastic_connection():
    
    es_connection = Elasticsearch('https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200/', 
                                  verify_certs = False,
                                  http_auth=('svc-bbm_aiml-user', 'v^*SRFP%uBM2v2bs9VG'))
    return es_connection


In [74]:
def elastic_data_generator(df, cols, index_name):
    for c, line in enumerate(df):
        yield {
          '_op_type': 'index',
          '_index': index_name,
          '_id': line.get('id'),
          'type': '_doc',
          '_source': 
            {k: line.get(k, ['No Data']) for k in cols}
        }


In [75]:
def create_index(es_connection, index_name):
    es_connection.indices.create(index=index_name, body=mappings)


In [76]:
def load_data_to_index(es_connection, df_dict, cols, index_name):
    
    if not df_dict:
        print("Empty dict - Nothing was indexed")
        return
    else:
        try:
            resp = helpers.bulk(es_connection, elastic_data_generator(df_dict, cols, index_name))
            print("finished uploading data")
        except Exception as e:
            print(e)

In [77]:
es_connection = initialize_elastic_connection()

In [79]:
load_data_to_index(es_connection, df_dict, all_cols, 'bbm_aiml_stm_test_prod_3')

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


finished uploading data


### Initialize Elastic index

In [137]:
es_connection = initialize_elastic_connection()
create_index(es_connection, 'bbm_aiml_stm_prod_v2')


C:\Users\ez99152\AppData\Local\Temp/ipykernel_13772/1666395428.py:2: DeprecationWarning: The 'body' parameter is deprecated for the 'create' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es_connection.indices.create(index=index_name, body=mappings)
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\AppData\Local\Temp/ipykernel_13772/1666395428.py:2: ElasticsearchWarning: The client is unable to verify that the server is Elasticsearch due security privileges on the server side
  es_connection.indices.create(index=index_name, body=mappings)
C:\Users\ez99152

## Query the data

In [29]:
q = {
    "fields": [
        "request_id",
        "accessPolicyTag",
        "workOrder.status",
        "workOrder.followUpDate",
        "workOrder.controlDesk",
        "workOrder.group ",
        "workOrder.assignee",
        "workOrder.tags",
        "source.orderDate",
        "source.requestedStartDate",
        "source.requestType",
        "source.requestSource",
        "source.customerSupportModel",
        "source.businessUnit",
        "source.status",
        "source.product",
        "source.serviceRegion",
        "source.goldenCustomer",
        "source.customer",
        "source.customerMarketSegment",
        "source.preferredLanguage",
        "source.focTarget"
  ]
}

In [50]:
q = make_elastic_query("2022-04-27T10:20:55")

In [80]:
r = es_connection.search(body=q, index='bbm_aiml_stm_test_prod_3', size=4000)

C:\Users\ez99152\AppData\Local\Temp/ipykernel_13772/793927149.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  r = es_connection.search(body=q, index='bbm_aiml_stm_test_prod_3', size=4000)
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [81]:
r

{'took': 98,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 260, 'relation': 'eq'},
  'max_score': 2.01131,
  'hits': [{'_index': 'bbm_aiml_stm_test_prod_3',
    '_type': '_doc',
    '_id': 'Fb-9s4ABhhVSglz-m5ck',
    '_score': 2.01131,
    '_source': {'request_id': 'bsd_REQ-544756',
     'lastUpdated': '2022-05-11T15:26:36.538190233Z',
     'accessPolicyTag': ['DIVISION_RETAIL'],
     'OrderResponse': {'value': 'no-data'},
     'workOrder': {'id': 'bsd_REQ-544756',
      'status': 'inProgress',
      'controlDesk': 'bbscL1',
      'assignee': {'id': 'ba0cdrq', 'name': 'Marc Larivee'},
      'tags': ['CREATED_FROM_SMARTPATH',
       'SMARTPATH_DISTRIBUTION_MODE_FILTER']},
     'source': {'id': 'bsd_REQ-544756',
      'externalId': 'REQ-544756',
      'orderDate': '2022-05-11T15:26:28Z',
      'requestedStartDate': '2022-05-11T15:26:28Z',
      'startDate': '2022-05-11T15:26:33.226889392Z',
      'lastUpdated': '20

In [83]:
len(r['hits']['hits'])

260

In [84]:
r['hits']['hits'][0]['fields']

{'source.customer': [{'name': 'Eurofins Enviroment Testing Canada'}],
 'source.customerSupportModel': ['Standard'],
 'accessPolicyTag': ['DIVISION_RETAIL'],
 'workOrder.status': ['inProgress'],
 'source.orderDate': ['2022-05-11T15:26:28Z'],
 'source.requestType': ['change'],
 'source.goldenCustomer': [{'name': 'Eurofins Enviroment Testing Canada',
   'id': '1004996774'}],
 'source.focTarget': [0],
 'source.requestSource': ['smartpath'],
 'workOrder.controlDesk': ['bbscL1'],
 'source.customerMarketSegment': ['Ontario SMB - Mass'],
 'lastUpdated': ['2022-05-11 15:26:36.53800+0000'],
 'source.status': ['inProgress'],
 'source.serviceRegion': ['ON'],
 'source.requestedStartDate': ['2022-05-11T15:26:28Z'],
 'source.preferredLanguage': ['en'],
 'source.businessUnit': ['Retail'],
 'workOrder.tags': ['CREATED_FROM_SMARTPATH',
  'SMARTPATH_DISTRIBUTION_MODE_FILTER'],
 'request_id': ['bsd_REQ-544756'],
 'workOrder.assignee': [{'name': 'Marc Larivee', 'id': 'ba0cdrq'}],
 'source.product': ['busin

In [37]:
def clean_resp_dict(resp):
    """Transform the response dict which can have nested fields into a flattened
    dict.
    Note: This works for current fields as of 04/22, and might need to be updated
    if more data fields are ingested."""
    
    cleaned_r_list = []

    for hit in resp['hits']['hits']:
        single_cleaned_r = {}
        for k, v in hit['fields'].items():
            if k == 'source.goldenCustomer' and 'id' in v[0].keys():
                single_cleaned_r[k] = v[0]['id']
            else:
                if len(v) == 1:
                    single_cleaned_r[k] = v[0]
                if len(v) > 1:
                    single_cleaned_r[k] = [' '.join(v)]
        cleaned_r_list.append(single_cleaned_r)

    return cleaned_r_list


In [38]:
cleaned_r_list =  clean_resp_dict(r)

In [39]:
df = pd.DataFrame(cleaned_r_list, columns=query_fields)

In [40]:
df.columns = [c.replace('.', '_') for c in query_fields]

In [41]:
df['source_requestedStartDate'] = pd.to_datetime(df.source_requestedStartDate)

In [44]:
df[(df.source_status != 'cancelled') & (df.source_status != 'completed')]

,request_id,accessPolicyTag,workOrder_status,workOrder_followUpDate,workOrder_controlDesk,workOrder_group,workOrder_assignee,workOrder_tags,source_orderDate,source_requestedStartDate,...,source_customerSupportModel,source_businessUnit,source_status,source_product,source_serviceRegion,source_goldenCustomer,source_customer,source_customerMarketSegment,source_preferredLanguage,source_focTarget
0,bsd_REQQA-468368,DIVISION_RETAIL,new,NaN,businessCare,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_DISTRIBU...,2022-04-07T18:09:44Z,2022-04-07 18:09:44+00:00,...,Standard,Retail,inProgress,NaN,EAST,NaN,NaN,Bell Atlantic,en,0
1,bsd_REQQA-468367,[EMAIL_WOA DIVISION_RETAIL],new,NaN,businessCare,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_DISTRIBU...,2022-04-07T18:09:42Z,2022-04-07 18:09:42+00:00,...,Standard,Retail,inProgress,NaN,EAST,NaN,NaN,Bell Atlantic,en,0
2,bsd_REQQA-468366,DIVISION_WIRELESS,new,NaN,hscToronto,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-07T18:08:39Z,2022-04-07 18:08:39+00:00,...,Standard,Wireless,acknowledged,NaN,NaN,NaN,NaN,NaN,en,0
3,bsd_REQQA-468365,DIVISION_WIRELESS,new,NaN,hscToronto,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-07T18:08:36Z,2022-04-07 18:08:36+00:00,...,Standard,Wireless,acknowledged,NaN,NaN,NaN,NaN,NaN,en,0
4,bsd_REQQA-468364,DIVISION_WIRELESS,new,NaN,hscToronto,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-07T18:08:32Z,2022-04-07 18:08:32+00:00,...,Standard,Wireless,acknowledged,NaN,NaN,NaN,NaN,NaN,en,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,bsd_REQQA-465564,DIVISION_WIRELESS,new,NaN,hscOttawa,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-06T15:23:17Z,2022-04-06 15:23:17+00:00,...,Standard,Wireless,acknowledged,NaN,NaN,NaN,NaN,NaN,en,0
3996,bsd_REQQA-465563,DIVISION_BMR,new,NaN,NaN,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_DISTRIBU...,2022-04-06T15:23:14Z,2022-04-06 15:23:14+00:00,...,Standard,BMR,acknowledged,NaN,NaN,NaN,NaN,Public Safety,en,0
3997,bsd_REQQA-465562,DIVISION_RETAIL,new,NaN,managedServicesSPOA,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-06T15:23:12Z,2022-04-06 15:23:12+00:00,...,Standard,Retail,acknowledged,managedServices,INTL,NaN,NaN,NaN,fr,0
3998,bsd_REQQA-465561,DIVISION_RETAIL,new,NaN,managedServicesSPOA,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-06T15:22:43Z,2022-04-06 15:22:43+00:00,...,Standard,Retail,acknowledged,businessLines,QC,NaN,NaN,NaN,fr,0


In [42]:
df.source_status.value_counts()

acknowledged    3004
inProgress       988
completed          5
pending            2
cancelled          1
Name: source_status, dtype: int64

In [88]:
df.workOrder_assignee.value_counts()

{'name': 'Julian Fernandes', 'id': 'ju6105298'}         14
{'name': 'Agnes Agent', 'id': 'agnes.agent'}             1
{'name': 'Donald Advanced', 'id': 'donald.advanced'}     1
{'name': 'Grant M Robinson', 'id': 'grant.robinson'}     1
Name: workOrder_assignee, dtype: int64

In [79]:
df.columns

Index(['request_id', 'accessPolicyTag', 'workOrder_status',
       'workOrder_followUpDate', 'workOrder_controlDesk', 'workOrder_group ',
       'workOrder_assignee', 'workOrder_tags', 'source_orderDate',
       'source_requestedStartDate', 'source_requestType',
       'source_requestSource', 'source_customerSupportModel',
       'source_businessUnit', 'source_status', 'source_product',
       'source_serviceRegion', 'source_goldenCustomer', 'source_customer',
       'source_customerMarketSegment', 'source_preferredLanguage',
       'source_focTarget'],
      dtype='object')

In [72]:
df.head()

,request_id,accessPolicyTag,workOrder_status,workOrder_followUpDate,workOrder_controlDesk,workOrder_group,workOrder_assignee,workOrder_tags,source_orderDate,source_requestedStartDate,...,source_customerSupportModel,source_businessUnit,source_status,source_product,source_serviceRegion,source_goldenCustomer,source_customer,source_customerMarketSegment,source_preferredLanguage,source_focTarget
0,bsd_REQQA-463570,DIVISION_WIRELESS,new,NaN,hscToronto,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-05T18:17:49Z,2022-04-05 18:17:49+00:00,...,Standard,Wireless,acknowledged,NaN,NaN,1004888274,{'name': 'Bank of Montreal'},Quebec Enterprise,en,0
1,bsd_REQQA-463569,DIVISION_BMR,new,NaN,NaN,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_DISTRIBU...,2022-04-05T18:17:19Z,2022-04-05 18:17:19+00:00,...,Standard,BMR,acknowledged,NaN,NaN,NaN,NaN,Public Safety,en,0
2,bsd_REQQA-463568,DIVISION_WHOLESALE,new,NaN,localMigration,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-05T18:17:17Z,2022-04-05 18:17:17+00:00,...,Standard,Wholesale,acknowledged,businessLines,ON,NaN,NaN,NaN,en,0
3,bsd_REQQA-463567,DIVISION_RETAIL,new,NaN,managedServicesSPOA,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-05T18:17:15Z,2022-04-05 18:17:15+00:00,...,Standard,Retail,acknowledged,managedServices,INTL,NaN,NaN,NaN,fr,0
4,bsd_REQQA-463566,DIVISION_WHOLESALE,new,NaN,wholesaleBroadband,NaN,NaN,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-04-05T18:17:12Z,2022-04-05 18:17:12+00:00,...,Standard,Wholesale,acknowledged,ethernet,ON,{},{},NaN,en,0


In [60]:
df[df.workOrder_tags.str.contains("SMARTPATH_DISTRIBUTION_MODE_FILTER")]

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [47]:
with open("tmp/test_file.txt") as file:
    d = {}
    for i , line in enumerate(file):
        print(line)
        print(type(line))
        if i == 3:
            break



{"agent_id":"jeffrey.sanderson","full_name":"Jeffrey Sanderson","permissions":["OC-SmartPath-NetNewCreator","OC-SmartPath-Wholesale","OC-SmartPath-Retail","OC-SmartPath-BBM"]}

<class 'str'>
{"agent_id":"ka61257","full_name":"Kathleen Lambert","permissions":["OC-SmartPath-Wholesale","OC-SmartPath-Retail","OC-SmartPath-BBM"]}

<class 'str'>
{"agent_id":"bccs.0122911","full_name":"Samantha Clarke","permissions":["OC-SmartPath-NetNewCreator","OC-SmartPath-Wholesale","OC-SmartPath-Retail","OC-SmartPath-BBM"]}

<class 'str'>
{"agent_id":"kim.chiodo","full_name":"Kim Chiodo","permissions":["OC-SmartPath-NetNewCreator","OC-SmartPath-Wholesale","OC-SmartPath-Retail","OC-SmartPath-BBM"]}

<class 'str'>


In [48]:
a = '{"agent_id":"jeffrey.sanderson","full_name":"Jeffrey Sanderson","permissions":["OC-SmartPath-NetNewCreator","OC-SmartPath-Wholesale","OC-SmartPath-Retail","OC-SmartPath-BBM"]}'
json.loads(a)

{'agent_id': 'jeffrey.sanderson',
 'full_name': 'Jeffrey Sanderson',
 'permissions': ['OC-SmartPath-NetNewCreator',
  'OC-SmartPath-Wholesale',
  'OC-SmartPath-Retail',
  'OC-SmartPath-BBM']}

In [54]:
with open("tmp/test_file.txt") as file:

    for i, line in enumerate(file):
        print(i)
        if i == 5:
            break
        d = json.loads(line)
        print(d['agent_id'])
        print(d['permissions'])
        print(d['full_name'])



0
jeffrey.sanderson
['OC-SmartPath-NetNewCreator', 'OC-SmartPath-Wholesale', 'OC-SmartPath-Retail', 'OC-SmartPath-BBM']
Jeffrey Sanderson
1
ka61257
['OC-SmartPath-Wholesale', 'OC-SmartPath-Retail', 'OC-SmartPath-BBM']
Kathleen Lambert
2
bccs.0122911
['OC-SmartPath-NetNewCreator', 'OC-SmartPath-Wholesale', 'OC-SmartPath-Retail', 'OC-SmartPath-BBM']
Samantha Clarke
3
kim.chiodo
['OC-SmartPath-NetNewCreator', 'OC-SmartPath-Wholesale', 'OC-SmartPath-Retail', 'OC-SmartPath-BBM']
Kim Chiodo
4
susan.grimsdale
['OC-SmartPath-GoC-Level1', 'OC-SmartPath-GoC-Level2', 'OC-SmartPath-NetNewCreator', 'OC-SmartPath-Wholesale', 'OC-SmartPath-Retail', 'OC-SmartPath-BBM']
Susan Grimsdale-Holder
5


In [53]:
d

{'jeffrey.sanderson': {'full_name': 'Jeffrey Sanderson',
  'permissions': ['OC-SmartPath-NetNewCreator',
   'OC-SmartPath-Wholesale',
   'OC-SmartPath-Retail',
   'OC-SmartPath-BBM']},
 'ka61257': {'full_name': 'Kathleen Lambert',
  'permissions': ['OC-SmartPath-Wholesale',
   'OC-SmartPath-Retail',
   'OC-SmartPath-BBM']},
 'bccs.0122911': {'full_name': 'Samantha Clarke',
  'permissions': ['OC-SmartPath-NetNewCreator',
   'OC-SmartPath-Wholesale',
   'OC-SmartPath-Retail',
   'OC-SmartPath-BBM']},
 'kim.chiodo': {'full_name': 'Kim Chiodo',
  'permissions': ['OC-SmartPath-NetNewCreator',
   'OC-SmartPath-Wholesale',
   'OC-SmartPath-Retail',
   'OC-SmartPath-BBM']},
 'susan.grimsdale': {'full_name': 'Susan Grimsdale-Holder',
  'permissions': ['OC-SmartPath-GoC-Level1',
   'OC-SmartPath-GoC-Level2',
   'OC-SmartPath-NetNewCreator',
   'OC-SmartPath-Wholesale',
   'OC-SmartPath-Retail',
   'OC-SmartPath-BBM']},
 'py6101652': {'full_name': 'Pyrave Thayananthan',
  'permissions': ['OC-Smar

In [33]:
a = json.loads(l2.strip('[]\n'))


In [35]:
a.pop('id', None)

'Ema'